# Experiments
This file contains the experiments that were used to find the optimal configuration of our LSTM model. The code of the model itself is located in [solution.py](solution.py).

Imports load_raw_data and inspect_dataset from solution.py and
prints the dataset summary. Run this to confirm the data is readable
and to learn input_size before building the model.

In [1]:
# %pip install --upgrade pip
# %pip uninstall -y torch torchvision torchaudio
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [2]:
import sys, torch
print(torch.__version__, )
print(f"Using device: {torch.device('cuda') if torch.cuda.is_available() else 'cpu'}, {torch.cuda.get_device_name(0) if torch.cuda.is_available() else ' --- '}")

2.5.1+cu121
Using device: cuda, NVIDIA GeForce RTX 4060 Laptop GPU


# Dataset

In [3]:
from solution import load_raw_data, inspect_dataset, TRAINING_DATA_PATH

training_data = load_raw_data(TRAINING_DATA_PATH)
inspect_dataset(training_data)

Samples             : 2939
Input size          : 1  (features per time step)
Min sequence length : 4
Max sequence length : 6308
First sequence      : 
	shape - (4756,) 
	label nr - 0 
	composer - bach
Composer class counts:
    0 (bach): 1630 samples
    1 (beethoven): 478 samples
    2 (debussy): 154 samples
    3 (scarlatti): 441 samples
    4 (victoria): 236 samples


Since the sequences can have various lengths (between 4 and 6308!) we need to to make their shape the same.

In [4]:
from torch.utils.data import DataLoader
from solution import ChordDataset, pad_batch


dataset = ChordDataset(training_data)
loader = DataLoader(
        dataset,
        batch_size=len(training_data),
        shuffle=True,
        collate_fn=pad_batch,
)

padded_sequences, labels, original_lengths = next(iter(loader))

print(f"Padded batch shape     : {padded_sequences.shape}")
print(f"First sequence (padded): {padded_sequences[0].shape}")
print(f"Original lengths       : {original_lengths[:30]}")

Padded batch shape     : torch.Size([2939, 6308, 1])
First sequence (padded): torch.Size([6308, 1])
Original lengths       : [676, 154, 179, 893, 375, 195, 68, 1373, 221, 288, 72, 84, 288, 52, 1230, 68, 282, 312, 714, 217, 164, 148, 60, 60, 51, 385, 56, 136, 129, 1696]


# Experiment evaluation helpers

In [5]:
import copy
import numpy as np
from sklearn.model_selection import StratifiedKFold
from solution import NetConfig, LSTMTrainer, TRAINING_DATA_PATH


def calc_balanced_accuracy(predictions: np.ndarray, targets: np.ndarray) -> float:
    """Mean per-class recall (balanced accuracy)."""
    return float(np.mean([
        (predictions[targets == class_id] == class_id).mean()
        for class_id in range(5)
        if (targets == class_id).sum() > 0
    ]))


def evaluate_cv(
    experiment_name: str,
    config: NetConfig,
    data: list,
    num_folds: int = 5,
) -> None:
    """Train and evaluate a single NetConfig using StratifiedKFold."""
    all_labels = np.array([label for _, label in data])
    fold_splitter = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
    fold_scores = []

    for train_indices, val_indices in fold_splitter.split(np.zeros(len(all_labels)), all_labels):
        print(".", end=" ")
        
        train_fold = [data[i] for i in train_indices]
        val_fold   = [data[i] for i in val_indices]
        val_labels = all_labels[val_indices]

        fold_trainer = LSTMTrainer(copy.deepcopy(config))
        fold_trainer.fit(train_fold)
        fold_predictions = fold_trainer.predict(val_fold)

        fold_scores.append(calc_balanced_accuracy(fold_predictions, val_labels))

    mean_score, std_score = np.mean(fold_scores), np.std(fold_scores)
    print(f"{experiment_name:50s}  acc = {mean_score:.4f} ± {std_score:.4f}")

# Experiments

### Hidden size

In [ ]:
for hidden_size in [32, 64, 128, 256, 512]:
    config = NetConfig(
        hidden_size=hidden_size,
        num_layers=2,
        dropout=0.3,
        epochs=25,
        bidirectional=False,
    )
    evaluate_cv(f"hidden={hidden_size} layers=2 dropout=0.3", config, training_data)

. . . . . hidden=32 layers=2 dropout=0.3                      acc = 0.4270 ± 0.0736
. . . . . hidden=64 layers=2 dropout=0.3                      acc = 0.5412 ± 0.0480
. . . . . hidden=128 layers=2 dropout=0.3                     acc = 0.5665 ± 0.0329
. . . . . hidden=256 layers=2 dropout=0.3                     acc = 0.6104 ± 0.0192


. . . . . hidden=32 layers=2 dropout=0.3                      **acc = 0.4270 ± 0.0736** <br/>
. . . . . hidden=64 layers=2 dropout=0.3                      **acc = 0.5412 ± 0.0480** <br/>
. . . . . hidden=128 layers=2 dropout=0.3                     **acc = 0.5665 ± 0.0329** <br/>
. . . . . hidden=256 layers=2 dropout=0.3                     **acc = 0.6104 ± 0.0192** <br/>

### Number of layers

In [7]:
for num_layers in [1, 2, 3]:
    config = NetConfig(
        hidden_size=64,
        num_layers=num_layers,
        dropout=0.3,
        epochs=15,
        bidirectional=False,
    )
    evaluate_cv(f"hidden=64 layers={num_layers} dropout=0.3", config, training_data)

. . . . . hidden=64 layers=1 dropout=0.3                      acc = 0.4747 ± 0.0269
. . . . . hidden=64 layers=2 dropout=0.3                      acc = 0.4551 ± 0.0503
. . . . . hidden=64 layers=3 dropout=0.3                      acc = 0.4730 ± 0.0308


. . . . . hidden=64 layers=1 dropout=0.3                      **acc = 0.4747 ± 0.0269** <br/>
. . . . . hidden=64 layers=2 dropout=0.3                      **acc = 0.4551 ± 0.0503** <br/>
. . . . . hidden=64 layers=3 dropout=0.3                      **acc = 0.4730 ± 0.0308** <br/>

### Dropout

In [8]:
for dropout in [0.0, 0.2, 0.3, 0.5]:
    config = NetConfig(
        hidden_size=64,
        num_layers=2,
        dropout=dropout,
        epochs=15,
        bidirectional=False,
    )
    evaluate_cv(f"hidden=64 layers=2 dropout={dropout}", config, training_data)

. . . . . hidden=64 layers=2 dropout=0.0                      acc = 0.4529 ± 0.0713
. . . . . hidden=64 layers=2 dropout=0.2                      acc = 0.4981 ± 0.0212
. . . . . hidden=64 layers=2 dropout=0.3                      acc = 0.4754 ± 0.0204
. . . . . hidden=64 layers=2 dropout=0.5                      acc = 0.4506 ± 0.0105


. . . . . hidden=64 layers=2 dropout=0.0                      **acc = 0.4529 ± 0.0713** <br/>
. . . . . hidden=64 layers=2 dropout=0.2                      **acc = 0.4981 ± 0.0212** <br/>
. . . . . hidden=64 layers=2 dropout=0.3                      **acc = 0.4754 ± 0.0204** <br/>
. . . . . hidden=64 layers=2 dropout=0.5                      **acc = 0.4506 ± 0.0105** <br/>

### Bidirectionality

In [ ]:
for bidir in [False, True]:
    config = NetConfig(
        hidden_size=64,
        num_layers=2,
        dropout=0.3,
        epochs=15,
        bidirectional=bidir,
    )
    evaluate_cv(f"hidden=64 layers=2 dropout=0.3 bidir={bidir}", config, training_data)

. . 

. . . . . hidden=64 layers=2 dropout=0.3 bidir=False          **acc = 0.4816 ± 0.0320** <br/>
. . . . . hidden=64 layers=2 dropout=0.3 bidir=True           **acc = 0.5258 ± 0.0430** <br/>

### Class weights and sampling experiments

Since the dataset is quite unbalanced, it could be beneficial to consider using a method to counter that. Here we will be comparing 3 methods and a base model to see which performs best.

In [8]:
configs = [
    (   "base",
        NetConfig(hidden_size=64,num_layers=2,dropout=0.3,epochs=15,bidirectional=False,batch_size=32)),
    
    (   "class_weights",
        NetConfig(hidden_size=64,num_layers=2,dropout=0.3,epochs=15,bidirectional=False,batch_size=32,use_class_weights=True)),
    
    (   "oversample",
        NetConfig( hidden_size=64, num_layers=2, dropout=0.3, epochs=15, bidirectional=False, batch_size=32, balance_strategy="oversample" )),
        
    (  "undersample",
        NetConfig(hidden_size=64,num_layers=2,dropout=0.3,epochs=15,bidirectional=False,batch_size=32,balance_strategy="undersample")),
]

for name, config in configs:
    evaluate_cv(f"experiment={name}", config, training_data)

. . . . . experiment=base                                     acc = 0.4712 ± 0.0272
. . . . . experiment=class_weights                            acc = 0.5343 ± 0.0254
. . . . . experiment=oversample                               acc = 0.5615 ± 0.0236
. . . . . experiment=undersample                              acc = 0.4619 ± 0.0163


. . . . . experiment=base                                     **acc = 0.4712 ± 0.0272** <br/>
. . . . . experiment=class_weights                            **acc = 0.5343 ± 0.0254** <br/>
. . . . . experiment=oversample                               **acc = 0.5615 ± 0.0236** <br/>
. . . . . experiment=undersample                              **acc = 0.4619 ± 0.0163** <br/>

In [ ]:
# TODO: sprawdź attention=True
# TODO: sprawdź scheduler_type="step"/"cosine"/"plateau" osobno
# TODO: grid search parametrów

# TODO: najlepszą konfigurację wykorzystać (tu lub w funkcji main solution.py) do wygenerowania finalnych predykcji na danych testowych

### Attention
### Scheduler

### Grid Search